# Gemini RAG Practice Notebook

Goal: understand how your own text/image data is retrieved from FAISS, then passed to Google Gemini so Gemini can generate an answer using your context.

ANS:

> My project uses CLIP embeddings and FAISS for retrieval. After retrieval, I pass the top matching documents as context to Gemini. Gemini does not search my files directly; my backend retrieves my data first, then Gemini generates the final answer from that retrieved context.

## Step 0: Install required packages

Run this only once if packages are missing.

In [1]:
# If this gives errors, run it once, then restart the notebook kernel.
%pip install -r requirements.txt

  Using cached fastapi-0.136.3-py3-none-any.whl.metadata (27 kB)
  Using cached python_multipart-0.0.30-py3-none-any.whl.metadata (2.1 kB)
  Using cached python_dotenv-1.2.2-py3-none-any.whl.metadata (27 kB)
  Using cached starlette-1.2.1-py3-none-any.whl.metadata (6.3 kB)
  Using cached pydantic-2.13.4-py3-none-any.whl.metadata (109 kB)
  Using cached typing_inspection-0.4.2-py3-none-any.whl.metadata (2.6 kB)
  Using cached distro-1.9.0-py3-none-any.whl.metadata (6.8 kB)
  Using cached sniffio-1.3.1-py3-none-any.whl.metadata (3.9 kB)
  Using cached annotated_types-0.7.0-py3-none-any.whl.metadata (15 kB)
  Using cached google_auth-2.53.0-py3-none-any.whl.metadata (5.5 kB)
  Using cached requests-2.34.2-py3-none-any.whl.metadata (4.8 kB)
  Using cached tenacity-9.1.4-py3-none-any.whl.metadata (1.2 kB)
  Using cached pyasn1_modules-0.4.2-py3-none-any.whl.metadata (3.5 kB)
  Using cached cryptography-48.0.0-cp311-abi3-win_amd64.whl.metadata (4.3 kB)
  Using cached urllib3-2.7.0-py3-none-a

## Step 1: Add your Gemini API key safely

Create a file named `.env` inside the `backend` folder.

Inside `backend/.env`, write:

```env
GEMINI_API_KEY=your_new_gemini_api_key_here
```

Important: do not paste real API keys into notebook cells. Do not commit `.env` to Git.

In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

has_key = bool(os.getenv("GEMINI_API_KEY"))
print("Gemini key loaded:", has_key)

Gemini key loaded: True


## Step 2: Load your FAISS vector store

Your indexed data is stored in `backend/store`:

- `index.faiss` stores vectors
- `metadata.json` stores text/image details
- `ids.json` connects FAISS vector positions to metadata IDs

In [2]:
from vector_store import FaissStore

store = FaissStore(path="store")
print("Total indexed items:", store.count())

Total indexed items: 21


## Step 3: Ask a question and retrieve matching data

This is retrieval only. Gemini is not used yet.

The question is converted into a CLIP text embedding. FAISS compares that query vector with stored text/image vectors and returns the closest matches.

In [3]:
from embeddings import embed_text

question = "What is RAG and how does this project use it?"

query_vector = embed_text(question)
hits = store.search(query_vector, top_k=3)

print("Question:", question)
print("Query vector shape:", query_vector.shape)
print("Retrieved items:", len(hits))

c:\Users\aarav\Desktop\AI-Powered Search Engine\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 398/398 [00:00<00:00, 2763.47it/s]


Question: What is RAG and how does this project use it?
Query vector shape: (512,)
Retrieved items: 3


In [4]:
for i, hit in enumerate(hits, start=1):
    meta = hit["metadata"]
    print(f"\nResult {i}")
    print("Score:", hit["score"])
    print("Title:", meta.get("title"))
    print("Type:", meta.get("type"))
    print("Source:", meta.get("source") or meta.get("filename"))
    print("Preview:", (meta.get("content") or meta.get("caption") or "")[:300])


Result 1
Score: 46.80170440673828
Title: secret_project
Type: text
Source: secret_project.txt
Preview: The secret project code is DRAGON-2025.

Result 2
Score: 52.914005279541016
Title: ml
Type: text
Source: ml.txt
Preview: Machine Learning is a subset of Artificial Intelligence that enables computers to learn patterns from data and make predictions without being explicitly programmed. Machine learning algorithms are commonly categorized into supervised learning, unsupervised learning, and reinforcement learning. Appli

Result 3
Score: 52.914005279541016
Title: ml
Type: text
Source: ml.txt
Preview: Machine Learning is a subset of Artificial Intelligence that enables computers to learn patterns from data and make predictions without being explicitly programmed. Machine learning algorithms are commonly categorized into supervised learning, unsupervised learning, and reinforcement learning. Appli


## Step 4: Convert retrieved results into context

Gemini cannot magically read your FAISS database. We must prepare text context and send it in the prompt.

For text documents, context comes from `metadata.content`.

For images, context comes from `metadata.caption`, `metadata.title`, and filename. In this beginner version, Gemini is not viewing the original image again; it is answering from image metadata/captions.

In [5]:
def build_context(retrieved_items):
    blocks = []

    for index, item in enumerate(retrieved_items, start=1):
        meta = item.get("metadata", {})
        title = meta.get("title", "Untitled")
        item_type = meta.get("type", "unknown")
        source = meta.get("source") or meta.get("filename") or "unknown"
        content = meta.get("content") or meta.get("caption") or ""

        blocks.append(
            f"Source {index}\n"
            f"Title: {title}\n"
            f"Type: {item_type}\n"
            f"File: {source}\n"
            f"Content: {content}"
        )

    return "\n\n---\n\n".join(blocks)

context = build_context(hits)
print(context[:1500])

Source 1
Title: secret_project
Type: text
File: secret_project.txt
Content: The secret project code is DRAGON-2025.

---

Source 2
Title: ml
Type: text
File: ml.txt
Content: Machine Learning is a subset of Artificial Intelligence that enables computers to learn patterns from data and make predictions without being explicitly programmed. Machine learning algorithms are commonly categorized into supervised learning, unsupervised learning, and reinforcement learning. Applications include spam detection, image recognition, recommendation systems, and predictive analytics.

---

Source 3
Title: ml
Type: text
File: ml.txt
Content: Machine Learning is a subset of Artificial Intelligence that enables computers to learn patterns from data and make predictions without being explicitly programmed. Machine learning algorithms are commonly categorized into supervised learning, unsupervised learning, and reinforcement learning. Applications include spam detection, image recognition, recommendation s

## Step 5: Build the prompt for Gemini

This is the most important RAG idea:

We tell Gemini to answer using only the retrieved context.

In [6]:
prompt = f"""
You are a helpful RAG assistant.

Answer the user's question using only the retrieved context below.
If the answer is not present in the context, say:
"I don't know based on the indexed documents."

User question:
{question}

Retrieved context:
{context}
"""

print(prompt[:2000])


You are a helpful RAG assistant.

Answer the user's question using only the retrieved context below.
If the answer is not present in the context, say:
"I don't know based on the indexed documents."

User question:
What is RAG and how does this project use it?

Retrieved context:
Source 1
Title: secret_project
Type: text
File: secret_project.txt
Content: The secret project code is DRAGON-2025.

---

Source 2
Title: ml
Type: text
File: ml.txt
Content: Machine Learning is a subset of Artificial Intelligence that enables computers to learn patterns from data and make predictions without being explicitly programmed. Machine learning algorithms are commonly categorized into supervised learning, unsupervised learning, and reinforcement learning. Applications include spam detection, image recognition, recommendation systems, and predictive analytics.

---

Source 3
Title: ml
Type: text
File: ml.txt
Content: Machine Learning is a subset of Artificial Intelligence that enables computers to lear

## Step 6: Send the prompt to Gemini

Now Gemini generates the final answer. Notice: Gemini is using the prompt we created from your retrieved data.

In [8]:
from google import genai

api_key = os.getenv("GEMINI_API_KEY")

if not api_key:
    raise RuntimeError("Missing GEMINI_API_KEY. Add it to backend/.env and restart the notebook kernel.")

client = genai.Client(api_key=api_key)

response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=prompt,
)

print(response.text)

I don't know based on the indexed documents.


## Step 7: Practice with your own questions

Change the question below and run the cell.

Try questions like:

- What is machine learning?
- What is the secret project code?
- Which images are about dogs?
- What does Aarav's project do?
- What projects are in Manish's portfolio?

In [9]:
def ask_my_data(user_question, top_k=3):
    query_vector = embed_text(user_question)
    retrieved = store.search(query_vector, top_k=top_k)
    rag_context = build_context(retrieved)

    rag_prompt = f"""
You are a helpful RAG assistant.

Answer the user's question using only the retrieved context below.
If the answer is not present in the context, say:
"I don't know based on the indexed documents."

User question:
{user_question}

Retrieved context:
{rag_context}
"""

    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=rag_prompt,
    )

    return response.text, retrieved


answer, sources = ask_my_data("What is the secret project code?", top_k=3)
print("ANSWER:\n", answer)

print("\nSOURCES USED:")
for source in sources:
    meta = source["metadata"]
    print("-", meta.get("title"), "|", meta.get("type"), "|", meta.get("source") or meta.get("filename"))

ANSWER:
 The secret project code is DRAGON-2025.

SOURCES USED:
- secret_project | text | secret_project.txt
- ml | text | ml.txt
- ml | text | ml.txt


## Interview Explanation

Use this explanation:

1. I store my documents and image metadata in a local data folder.
2. I use CLIP embeddings to convert text and images into 512-dimensional vectors.
3. I store those vectors in FAISS for fast similarity search.
4. When a user asks a question, I convert the question into an embedding.
5. FAISS returns the top matching documents/images from my own indexed data.
6. I format those retrieved results as context.
7. I send the user question plus retrieved context to Gemini.
8. Gemini generates the final answer, but the answer is grounded in my retrieved data.

Short version:

> FAISS retrieves the right information from my own data, and Gemini turns that retrieved context into a natural-language answer.